In [61]:
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib

import sklearn


from copy import deepcopy
random_state: int = 42

pd.set_option('display.max_columns', None)

## import local lib

import api.utils



In [26]:
PROJECT_ROOT = pathlib.Path.cwd().parent
data_path = PROJECT_ROOT / 'data' / 'processed' / 'processed_data.csv'

In [28]:
_processed_df = pd.read_csv( data_path )
_processed_df.shape

(8849, 12)

In [29]:
processed_df = deepcopy( _processed_df )
processed_df = processed_df.drop( columns= ['start_time', 'end_time', 'modified_date'], errors= 'ignore'  )

processed_df.head(2)


,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,source_type,state,area,pop2020
0,214000.0,0.513,138.0,0.345,417000.0,gas,Alabama,12.899239,5024279.0
1,204000.0,0.513,138.0,0.329,398000.0,gas,Alabama,12.899239,5024279.0


In [ ]:
## setting proper dtypes
num_cols = [
    'emissions_quantity', 'capacity', 'capacity_factor', 'activity',
    'area', 'pop2020'
]

cat_cols = ['state', 'source_type']

## create a dtype dict based on above lists




for c in num_cols:
    if c in processed_df.columns:
        processed_df[c] = pd.to_numeric(processed_df[c], errors='coerce')

for c in cat_cols:
    if c in processed_df.columns:
        processed_df[c] = processed_df[c].astype('string').str.strip()


In [31]:
processed_df[ num_cols ].describe()
processed_df.head()

,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,source_type,state,area,pop2020
0,214000.0,0.513,138.0,0.345,417000.0,gas,Alabama,12.899239,5024279.0
1,204000.0,0.513,138.0,0.329,398000.0,gas,Alabama,12.899239,5024279.0
2,209000.0,0.515,138.0,0.335,406000.0,gas,Alabama,12.899239,5024279.0
3,201000.0,0.513,138.0,0.324,392000.0,gas,Alabama,12.899239,5024279.0
4,113000.0,0.452,87.0,0.328,250000.0,gas,Texas,65.363350,29145505.0


In [38]:
'  asd  a aa'.replace( ' ', '' )
processed_df['state'].unique()
processed_df['source_type'].unique()


<StringArray>
['gas', 'oil', 'coal', 'other_fossil', 'biomass', 'waste']
Length: 6, dtype: string

In [75]:

feature_df = (  deepcopy( processed_df )
	# 1) transforms + ratios 
	.assign(
		log1p_activity=         lambda df:  np.log1p(  df['activity']  ),
		log1p_capacity=         lambda df:  np.log1p(  df['capacity']  ),
		log1p_pop2020=          lambda df:  np.log1p(  df['pop2020']  ),
		log1p_area=             lambda df:  np.log1p(  df['area']  ),
		log1Pop_density=      		lambda df:  np.log1p(  df['pop2020'] / df['area'].replace(  0, np.nan  )  ),
		activity_per_capita=    lambda df:  df['activity'] / df['pop2020'].replace(  0, np.nan  ),
		activity_per_area=      lambda df:  df['activity'] / df['area'].replace(  0, np.nan  ),
		capacity_per_capita=    lambda df:  df['capacity'] / df['pop2020'].replace(  0, np.nan  ),
		capacity_density=       lambda df:  df['capacity'] / df['area'].replace(  0, np.nan  ),

	# 2) power-system structure features 
		potential_output=               lambda df:  df['capacity'] * df['capacity_factor'],
		utilization_ratio=              lambda df:  df['activity'] / (  (df['capacity'] * df['capacity_factor']).replace(  0, np.nan  )  ),
		activity_capacityFactor=     lambda df:  df['activity'] * df['capacity_factor'],
		activity_per_capacity=          lambda df:  df['activity'] / df['capacity'].replace(  0, np.nan  ),
		activity_capacity=      lambda df:  df['activity'] * df['log1p_capacity'],
		capacity_factor_capacity=       lambda df: df['capacity_factor'] * df['log1p_capacity'],
	## 3. interaction features
		state =  lambda df: df['state'].str.replace( ' ', '' ),
		source_type =  lambda df: df['source_type'].str.replace( '_', '' ),
		inter =  lambda df: df.apply( lambda _df: f"{_df['state']}_{_df['source_type']}", axis= 'columns'   ) ,
	
	)
    ## One-hot encoding for state & interaction-field
    .pipe( api.utils.OHE_func, categorical_col= ['state', 'inter']  )
    
    
	.drop( columns= ['emissions_factor'] ) ## as using this field would leakage the data
)
feature_df.shape
feature_df.head()
 

,emissions_quantity,capacity,capacity_factor,activity,source_type,area,pop2020,log1p_activity,log1p_capacity,log1p_pop2020,log1p_area,log1Pop_density,activity_per_capita,activity_per_area,capacity_per_capita,capacity_density,potential_output,utilization_ratio,activity_capacityFactor,activity_per_capacity,activity_capacity,capacity_factor_capacity,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_DistrictofColumbia,state_Florida,state_Georgia,state_Idaho,state_Illinois,state_Indiana,state_Iowa,state_Kansas,state_Kentucky,state_Louisiana,state_Maine,state_Maryland,state_Massachusetts,state_Michigan,state_Minnesota,state_Mississippi,state_Missouri,state_Montana,state_Nebraska,state_Nevada,state_NewHampshire,state_NewJersey,state_NewMexico,state_NewYork,state_NorthCarolina,state_NorthDakota,state_Ohio,state_Oklahoma,state_Oregon,state_Pennsylvania,state_RhodeIsland,state_SouthCarolina,state_SouthDakota,state_Tennessee,state_Texas,state_Utah,state_Vermont,state_Virginia,state_Washington,state_WestVirginia,state_Wisconsin,state_Wyoming,inter_Alabama_coal,inter_Alabama_gas,inter_Alabama_oil,inter_Arizona_coal,inter_Arizona_gas,inter_Arizona_oil,inter_Arkansas_biomass,inter_Arkansas_coal,inter_Arkansas_gas,inter_California_coal,inter_California_gas,inter_California_oil,inter_California_otherfossil,inter_California_waste,inter_Colorado_coal,inter_Colorado_gas,inter_Colorado_oil,inter_Connecticut_coal,inter_Connecticut_gas,inter_Connecticut_oil,inter_Connecticut_waste,inter_Delaware_coal,inter_Delaware_gas,inter_Delaware_oil,inter_DistrictofColumbia_gas,inter_Florida_coal,inter_Florida_gas,inter_Florida_oil,inter_Florida_otherfossil,inter_Florida_waste,inter_Georgia_biomass,inter_Georgia_coal,inter_Georgia_gas,inter_Georgia_oil,inter_Georgia_otherfossil,inter_Idaho_gas,inter_Idaho_otherfossil,inter_Illinois_coal,inter_Illinois_gas,inter_Illinois_oil,inter_Indiana_coal,inter_Indiana_gas,inter_Indiana_oil,inter_Indiana_otherfossil,inter_Iowa_coal,inter_Iowa_gas,inter_Iowa_oil,inter_Iowa_otherfossil,inter_Kansas_coal,inter_Kansas_gas,inter_Kansas_oil,inter_Kentucky_coal,inter_Kentucky_gas,inter_Kentucky_oil,inter_Louisiana_biomass,inter_Louisiana_coal,inter_Louisiana_gas,inter_Louisiana_otherfossil,inter_Louisiana_waste,inter_Maine_biomass,inter_Maine_gas,inter_Maine_oil,inter_Maine_waste,inter_Maryland_coal,inter_Maryland_gas,inter_Maryland_oil,inter_Maryland_waste,inter_Massachusetts_gas,inter_Massachusetts_oil,inter_Massachusetts_waste,inter_Michigan_coal,inter_Michigan_gas,inter_Michigan_oil,inter_Michigan_otherfossil,inter_Michigan_waste,inter_Minnesota_coal,inter_Minnesota_gas,inter_Minnesota_oil,inter_Minnesota_waste,inter_Mississippi_coal,inter_Mississippi_gas,inter_Mississippi_oil,inter_Missouri_coal,inter_Missouri_gas,inter_Missouri_oil,inter_Montana_coal,inter_Montana_gas,inter_Montana_otherfossil,inter_Nebraska_coal,inter_Nebraska_gas,inter_Nebraska_oil,inter_Nevada_coal,inter_Nevada_gas,inter_NewHampshire_coal,inter_NewHampshire_gas,inter_NewHampshire_oil,inter_NewHampshire_waste,inter_NewJersey_coal,inter_NewJersey_gas,inter_NewJersey_oil,inter_NewJersey_otherfossil,inter_NewJersey_waste,inter_NewMexico_coal,inter_NewMexico_gas,inter_NewMexico_oil,inter_NewMexico_otherfossil,inter_NewYork_coal,inter_NewYork_gas,inter_NewYork_oil,inter_NewYork_waste,inter_NorthCarolina_coal,inter_NorthCarolina_gas,inter_NorthCarolina_oil,inter_NorthCarolina_otherfossil,inter_NorthDakota_coal,inter_NorthDakota_gas,inter_Ohio_biomass,inter_Ohio_coal,inter_Ohio_gas,inter_Ohio_oil,inter_Ohio_otherfossil,inter_Oklahoma_coal,inter_Oklahoma_gas,inter_Oklahoma_oil,inter_Oklahoma_waste,inter_Oregon_coal,inter_Oregon_gas,inter_Oregon_waste,inter_Pennsylvania_biomass,inter_Pennsylvania_coal,inter_Pennsylvania_gas,inter_Pennsylvania_oil,inter_Pennsylvania_otherfossil,inter_Pennsylvania_waste,inter_RhodeIsland_gas,inter_SouthCarolina_coal,inter_SouthCarolina_gas,inter_SouthCarolina_oil,inter_SouthDakota_coa